# Tutorial 8: Train NicheTrans on human lymph node data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_human_lymph_node import Lymph_node

from utils.utils import *
from utils.utils_training_human_lymph_node import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_human_lymph_node.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.1, n_source=3000, workers=4, adata_path='/mnt/datadisk0/Processed_DATA/2024_nm_human_lymph_nodes/', max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = Lymph_node(adata_path=args.adata_path, n_top_genes=args.n_source)
trainloader, testloader = human_node_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,
                   n_spot_types=dataset.n_spot_types)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

  Slice slice1: Leiden found 6 local clusters (sizes: [982, 891, 599, 536, 343, 133])
  Slice slice2: Leiden found 4 local clusters (sizes: [1608, 1257, 371, 123])
  Slice slice2: matched 4/4 local clusters into the global space
  Cross-slice alignment complete: 6 global cell types
------Calculating spatial graph...
The graph contains 13638 edges, 3484 cells.
3.9145 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 27174 edges, 3484 cells.
7.7997 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 13138 edges, 3359 cells.
3.9113 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 26192 edges, 3359 cells.
7.7976 neighbors per cell on average.
=> Human lymph node loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  After filting  3484 spots
  test     |  After filting  3359 spots
  ------------------------------
 

### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_human_lymph_node_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 108/108	 Loss 1.116628 (0.984592)
==> Epoch 2/20
Batch 108/108	 Loss 0.524217 (0.877436)
==> Epoch 3/20
Batch 108/108	 Loss 0.537341 (0.840778)
==> Epoch 4/20
Batch 108/108	 Loss 0.617085 (0.827427)
==> Epoch 5/20
Batch 108/108	 Loss 0.470816 (0.811684)
==> Epoch 6/20
Batch 108/108	 Loss 0.459989 (0.802599)
==> Epoch 7/20
Batch 108/108	 Loss 0.839235 (0.784330)
==> Epoch 8/20
Batch 108/108	 Loss 0.349572 (0.767134)
==> Epoch 9/20
Batch 108/108	 Loss 0.300330 (0.749185)
==> Epoch 10/20
Batch 108/108	 Loss 0.560081 (0.724415)
==> Epoch 11/20
Batch 108/108	 Loss 0.399369 (0.704446)
==> Epoch 12/20
Batch 108/108	 Loss 0.348562 (0.687189)
==> Epoch 13/20
Batch 108/108	 Loss 0.313897 (0.679705)
==> Epoch 14/20
Batch 108/108	 Loss 0.392410 (0.674360)
==> Epoch 15/20
Batch 108/108	 Loss 0.449155 (0.665271)
==> Epoch 16/20
Batch 108/108	 Loss 0.366448 (0.666173)
==> Epoch 17/20
Batch 108/108	 Loss 0.571299 (0.661977)
==> Epoch 18/20
Batch 108/108	 Loss 0.355468 (0.660009)
=